In [ ]:
import xarray as xr
from dask.distributed import Client as DaskClient
from dep_tools.loaders import OdcLoader
from dep_tools.namers import LocalPath
from dep_tools.searchers import PystacSearcher
from dep_tools.writers import write_to_local_storage

from src.run_task import MLProcessor, add_indices, get_tiles

/Users/wj/Projects/ldn-lulc/dep-ml-products/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Optionally set up a local dask cluster
client = DaskClient(
    n_workers=1,
    threads_per_worker=None,
    memory_limit=None,
)
client.dashboard_link

/Users/wj/Projects/ldn-lulc/dep-ml-products/.venv/lib/python3.14/site-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 56255 instead
  warnings.warn(


'http://127.0.0.1:56255/status'

In [12]:
# Study site configuration
tile_id = "63,20"
year = "2024"

# For this tile, only 2024 is available for the dep_s1_geomad.
# https://stac-browser.staging.digitalearthpacific.io/collections/dep_s1_geomad/items/dep_s1_geomad_063_020_2024
# Prod has 2017-2024 https://stac.prod.digitalearthpacific.io/collections/dep_s1_geomad/items/dep_s1_geomad_063_020_2022 

# Get the study site
tiles = get_tiles()
area = tiles.loc[[(tile_id)]]
# area.explore()

In [ ]:
# Find some items
searcher = PystacSearcher(
    catalog="https://stac.staging.digitalearthpacific.io",
    collections=["dep_s1_geomad", "dep_s2_geomad"],
    datetime=year
)

items = searcher.search(area)

items_by_collection = {}
for item in items:
    items_by_collection.setdefault(item.collection_id, []).append(item)

# This hits transient errors
dem_searcher = PystacSearcher(
    catalog="https://planetarycomputer.microsoft.com/api/stac/v1",
    collections=["cop-dem-glo-30"]
)
items_by_collection["cop-dem-glo-30"] = dem_searcher.search(area)

print({k: len(v) for k, v in items_by_collection.items()})

# TODO: https://stac.staging.digitalearthpacific.io/collections/dep_s1_mosaic/items is empty
# In prod dep_s1_mosaic isn't even a collection.  https://stac.prod.digitalearthpacific.io/collections/

{'dep_s2_geomad': 6, 'dep_s1_geomad': 6, 'cop-dem-glo-30': 4}


In [ ]:
data = None

# Set up a data loader
loader = OdcLoader(
    resolution=10,
    crs=3832,
    groupby="solar_day",
    chunks={"time": 1, "x": 4096, "y": 4096},
    fail_on_error=False
)

# Run the load process, which is lazy-loaded

all_data = [loader.load(items, area).squeeze("time") for items in items_by_collection.values()]

loaded = xr.merge(all_data, compat='override') # TODO: Here count from s1 and s2 silently break by dropping 1.
loaded = loaded.rename({"data": "elevation"})
# loaded = loaded.drop_vars(["vv", "vh", "stdev_vv", "stdev_vh", "count"]) # TODO: Figure out which bands are needed by the model. Need to find the code that made the model because it doesn't have feature names.
loaded = loaded.chunk({"x": 4096, "y": 4096})
loaded

/var/folders/68/g6_znfts0kg6hq81twk8hbpr0000gn/T/ipykernel_12703/4284449288.py:16: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'y' ('y',) The recommendation is to set join explicitly for this case.
  loaded = xr.merge(all_data, compat='override')
/var/folders/68/g6_znfts0kg6hq81twk8hbpr0000gn/T/ipykernel_12703/4284449288.py:16: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'x' ('x',) The recommendation is to set join explicitly for this case.
  loaded = xr.merge(all_data, compat='override')


<xarray.Dataset> Size: 73GB
Dimensions:      (y: 28800, x: 27625)
Coordinates:
  * y            (y) float64 230kB -2.176e+06 -2.176e+06 ... -1.888e+06
  * x            (x) float64 221kB 2.952e+06 2.952e+06 ... 3.228e+06 3.228e+06
    spatial_ref  int32 4B 3832
    time         datetime64[ns] 8B 2024-01-01
Data variables: (12/23)
    nir          (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    red          (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    blue         (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    emad         (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    smad         (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    bcmad        (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    ...           ...
    vv           (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    mean_vh      (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    mean_vv      (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    stdev_vh     (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    stdev_vv     (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    elevation    (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>

In [36]:
input_data = add_indices(loaded)
input_data

<xarray.Dataset> Size: 95GB
Dimensions:      (y: 28800, x: 27625)
Coordinates:
  * y            (y) float64 230kB -2.176e+06 -2.176e+06 ... -1.888e+06
  * x            (x) float64 221kB 2.952e+06 2.952e+06 ... 3.228e+06 3.228e+06
    spatial_ref  int32 4B 3832
    time         datetime64[ns] 8B 2024-01-01
Data variables: (12/30)
    nir          (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    red          (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    blue         (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    emad         (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    smad         (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    bcmad        (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    ...           ...
    mndwi        (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    evi          (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    savi         (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    bsi          (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    ndmi         (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
    ndbi         (y, x) float32 3GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>

In [ ]:
import joblib
loaded_model = joblib.load("../models/test_model_04092024.dump")
print(f"Model has {loaded_model.n_features_in_} features")
print(f"Feature names: {getattr(loaded_model, 'feature_names_in_', 'not recorded. The model was trained without named columns')}")

Model has 24 features
Feature names: not recorded — trained without named columns


In [37]:
model = joblib.load("../models/test_model_04092024.dump")
print(getattr(model, "feature_names_in_", "NOT RECORDED"))

NOT RECORDED


In [ ]:
# Set up a data processor
processor = MLProcessor(
    model_path="../models/test_model_04092024.dump", # New model expects 24 features, not 26
    chunk_size=None,
    load_data=True
)

# Plan the processing.
output_data = processor.process(input_data)
output_data

predicting...


ValueError: X has 26 features, but RandomForestClassifier is expecting 24 features as input.

In [ ]:
from matplotlib import colors

classes = [
    [1, "bare_land", "#968640"],
    [2, "forest", "#064a00"],
    [3, "crops", "#ffce33"],
    [4, "grassland", "#d7ffa0"],
    [5, "settlements", "#b3b2ae"],
    [6, "mangroves", "#07b28d"],
    [7, "water", "#71a8ff"],
    [8, "quarry", "#b03a2e"]
]

values_list = [c[0] for c in classes]
color_list = [c[2] for c in classes]

# Build a listed colormap.
c_map = colors.ListedColormap(color_list)
bounds = values_list + [9]
norm = colors.BoundaryNorm(bounds, c_map.N)

output_data["class"].plot.imshow(cmap=c_map, norm=norm, size=10)

In [ ]:
for var in output_data.data_vars:
    output_data[var].odc.write_cog(f"test_{var}_nadi.tif", overwrite=True)

In [ ]:
# Testing the Azure writer

# from dep_tools.writers import AzureDsWriter
# from dep_tools.namers import DepItemPath

# itempath = DepItemPath("geomad", "test", "0.0", datetime)

# writer = AzureDsWriter(
#     itempath=itempath,
#     overwrite=True,
#     convert_to_int16=False,
#     extra_attrs=dict(dep_version="0.0"),
# )

# writer.write(output_data, "test")


In [ ]:
# Testing the AWS writer

from dep_tools.namers import DepItemPath
from dep_tools.writers import AwsDsCogWriter

itempath = DepItemPath("geomad", "test", "0.0", datetime)

writer = AwsDsCogWriter(
    itempath=itempath,
    overwrite=False,
    convert_to_int16=False,
    extra_attrs={"dep_version": "0.0"},
    bucket="files.auspatious.com"
)

writer.write(output_data, "test")

In [ ]:
from odc.stac import load
from pystac import Item

item = Item.from_file("https://deppcpublicstorage.blob.core.windows.net/output/dep_geomad_test/0-0/test/2023-01/dep_geomad_test_test_2023-01.stac-item.json")

data = load([item], chunks={})
data

In [ ]:
# This is the target path
dep_path = LocalPath(
    local_folder="data",
    sensor="s1",
    dataset_id="mosaic",
    version="0.0.0",
    time=datetime,
    zero_pad_numbers=True
)

item_id = "example_item_id"

# Set up a writer and write out the files
writer = write_to_local_storage(
    d=output_data,
    path=str(dep_path),
    write_args={"nodata": 0},
    use_odc_writer=True,
    overwrite=True,
    # convert_to_int16=False # Not sure if this exists
)
print(f"Writing to: {dep_path._folder(item_id)}")
out_files = writer.write(output_data, item_id)

In [ ]:
# Make sure written files are readable
stac_path = dep_path.path(item_id, ext=".stac-item.json")

item = Item.from_file(stac_path)
item.validate()

In [ ]:
from odc.stac import configure_s3_access, load
from pystac import Item

configure_s3_access(cloud_defaults=True, aws_unsigned=True)
# TODO: replace dep_s1_mosaic with dep_s1_geomad?
item = Item.from_file("https://dep-public-test.s3.us-west-2.amazonaws.com/dep_s1_mosaic/0-0-3b/066/020/2023/dep_s1_mosaic_066_020_2023.stac-item.json")

data = load([item], chunks={})

In [ ]:
data.isel(time=0).mean_vv.plot.imshow(size=10, robust=True)